# Chess Transformer — GPU Training

Full training run for all 4 attention variants:
- **Vanilla MHA** — baseline with sinusoidal positional encoding
- **RoPE MHA** — rotary position embeddings (used in LLaMA, Gemma)
- **GQA** — grouped query attention, fewer KV heads (used in Mistral, LLaMA-2 70B)
- **Sliding Window** — local sparse attention (used in Longformer, Mistral)

Run on **GPU runtime** (Runtime → Change runtime type → T4 GPU).

In [ ]:
!git clone https://github.com/matiimonti/Transformer_project
%cd Transformer_project

In [ ]:
import urllib.request
import zstandard as zstd
import os
import sys
sys.path.insert(0, 'src')

!pip install chess zstandard wandb -q
import chess

import torch
from train import train, make_attention_factory
from benchmark import run_benchmark
from model import ChessTransformer
from pgn_data import ChessTokenizer
from scale import run_scaling
from visualize import plot_all_layers

# Verify GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device.upper()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found — training will be slow. Switch to GPU runtime.')

In [ ]:
import wandb
wandb.login()   # paste API key when prompted (get it from wandb.ai/authorize)

## 1. Data

In [ ]:
os.makedirs('data', exist_ok=True)

YEAR  = '2013'
MONTH = '01'
url      = f'https://database.lichess.org/standard/lichess_db_standard_rated_{YEAR}-{MONTH}.pgn.zst'
zst_path = f'data/lichess_{YEAR}_{MONTH}.pgn.zst'
pgn_path = 'data/games.pgn'

### Stream download
def download(url, dest):
    def reporthook(count, block_size, total_size):
        mb_done  = count * block_size / 1_000_000
        mb_total = total_size / 1_000_000 if total_size > 0 else '?'
        print(f'\r  {mb_done:.1f} MB / {mb_total} MB', end='', flush=True)
    urllib.request.urlretrieve(url, dest, reporthook=reporthook)
    print()

print(f'Downloading: {url}')
download(url, zst_path)

print('Decompressing...')
dctx = zstd.ZstdDecompressor()
with open(zst_path, 'rb') as compressed:
    with open(pgn_path, 'wb') as out_file:
        dctx.copy_stream(compressed, out_file)

print(f'PGN saved to: {pgn_path}')
print(f'File size:    {os.path.getsize(pgn_path) / 1_000_000:.1f} MB')

## 2. Train All Variants

Each variant trains for 3000 steps (~15-20 min per variant on T4).
Checkpoints and metrics are saved to `checkpoints/<variant>/`.

In [ ]:
BASE_CONFIG = {
    'pgn_path': pgn_path,
    'max_games': 50_000,
    'seq_len': 128,
    'train_split': 0.9,
    'd_model': 128,
    'n_heads': 4,
    'n_layers': 4,
    'dropout': 0.1,
    'kv_heads': 2,
    'window_size': 32,
    'batch_size': 64,
    'num_workers': 2,
    'max_steps': 3000,
    'warmup_steps': 300,
    'max_lr': 3e-4,
    'min_lr': 3e-5,
    'weight_decay': 0.1,
    'grad_clip': 1.0,
    'log_interval': 100,
    'eval_interval': 500,
    'patience': 5,            # early stopping: stop if no improvement for 5 evals
    'wandb': True,         # log metrics to Weights & Biases
    'compile': True,         # torch.compile() for faster GPU training
}

for variant in ['vanilla', 'rope', 'gqa', 'sparse']:
    print(f'\n{"="*60}')
    print(f'Training variant: {variant.upper()}')
    print(f'{"="*60}')
    config = {**BASE_CONFIG, 'variant': variant, 'out_dir': f'checkpoints/{variant}'}
    train(config)

print('\nAll variants trained!')

## 3. Benchmark

In [ ]:
benchmark_config = {
    'd_model': BASE_CONFIG['d_model'],
    'n_heads': BASE_CONFIG['n_heads'],
    'n_layers': BASE_CONFIG['n_layers'],
    'seq_len': BASE_CONFIG['seq_len'],
    'kv_heads': BASE_CONFIG['kv_heads'],
    'window_size': BASE_CONFIG['window_size'],
    'dropout': BASE_CONFIG['dropout'],
}

run_benchmark('checkpoints', benchmark_config)

In [ ]:
from IPython.display import Image, display

display(Image('plots/loss_curves.png'))
display(Image('plots/benchmark.png'))

## 4. Generate Sample Games

Load the best checkpoint and sample move sequences autoregressively.

In [ ]:
BEST_VARIANT = 'rope'   # change to whichever had lowest val_loss

ckpt = torch.load(f'checkpoints/{BEST_VARIANT}/best_model.pt', map_location=device)
tokenizer = ChessTokenizer.load(f'checkpoints/{BEST_VARIANT}/tokenizer.json')
cfg = ckpt['config']

factory = make_attention_factory(cfg)
model = ChessTransformer(
    vocab_size = tokenizer.vocab_size,
    attention_factory = factory,
    d_model = cfg['d_model'],
    n_heads = cfg['n_heads'],
    n_layers = cfg['n_layers'],
    use_sinusoidal_pe = (BEST_VARIANT != 'rope'),
).to(device)
model.load_state_dict(ckpt['model_state'])

print(f'Loaded {BEST_VARIANT} checkpoint (val_loss={ckpt["val_loss"]:.4f})')
print(f'Vocab size: {tokenizer.vocab_size}\n')

seed = torch.tensor([[tokenizer.bos_id]], device=device)

for i in range(5):
    out = model.generate(seed, max_new_tokens=40, temperature=0.8, top_k=40)
    moves = tokenizer.decode(out[0].tolist()[1:])          # skip BOS
    moves = [m for m in moves if m not in ('<EOS>', '<PAD>', '<UNK>')]
    print(f'Game {i+1}: {" ".join(moves)}')

## 5. Move Legality Evaluation

Check what fraction of generated moves are legal according to chess rules,
using python-chess for full rule verification.

In [ ]:
def eval_move_legality(model, tokenizer, device, n_games=100):
    """
    Generates n_games sequences and measures the fraction of moves
    that are legal according to full chess rules via python-chess.

    Stops each game at the first illegal move — only moves up to
    that point count as legal. Matches the metric used in chess LM papers.
    """
    legal_count = 0
    total_count = 0
    seed = torch.tensor([[tokenizer.bos_id]], device=device)

    model.eval()
    with torch.no_grad():
        for _ in range(n_games):
            board = chess.Board()
            generated = model.generate(seed, max_new_tokens=40, temperature=1.0, top_k=40)
            moves = tokenizer.decode(generated[0].tolist()[1:])  # skip BOS

            for move_str in moves:
                if move_str in ('<EOS>', '<PAD>'):
                    break
                if move_str in ('<BOS>', '<UNK>'):
                    total_count += 1
                    break
                total_count += 1
                try:
                    move = board.parse_san(move_str)
                    board.push(move)
                    legal_count += 1
                except Exception:
                    break   # illegal or ambiguous — stop this game

    return legal_count / max(total_count, 1)


# Evaluate all variants
print(f'{"Variant":<16} {"Legality":>10}')
print('-' * 28)

for variant in ['vanilla', 'rope', 'gqa', 'sparse']:
    ckpt_path = f'checkpoints/{variant}/best_model.pt'
    tok_path = f'checkpoints/{variant}/tokenizer.json'

    if not __import__('os').path.exists(ckpt_path):
        print(f'{variant:<16} not trained yet')
        continue

    ckpt = torch.load(ckpt_path, map_location=device)
    tok = ChessTokenizer.load(tok_path)
    cfg = ckpt['config']
    factory = make_attention_factory(cfg)

    m = ChessTransformer(
        vocab_size = tok.vocab_size,
        attention_factory = factory,
        d_model = cfg['d_model'],
        n_heads = cfg['n_heads'],
        n_layers = cfg['n_layers'],
        max_seq_len = cfg['seq_len'],
        use_sinusoidal_pe = (variant != 'rope'),
    ).to(device)
    m.load_state_dict(ckpt['model_state'])

    legality = eval_move_legality(m, tok, device, n_games=100)
    print(f'{variant:<16} {legality:>9.1%}')

## 6. Scaling Law Experiment

Train three model sizes (1M / 5M / 20M parameters) on the same data and plot
validation loss against cumulative training FLOPs.

If loss follows a power law in compute — `L ∝ C^b` — the log-log plot will show
roughly parallel straight lines, consistent with the Chinchilla scaling result.

In [ ]:
scaling_args = {
    'pgn_path': pgn_path,
    'max_games': 50_000,
    'max_steps': 3000,      # steps per model size (~20-30 min total on T4)
    'batch_size': 128,       # larger batch uses GPU memory efficiently
    'num_workers': 2,
    'log_interval': 100,
    'eval_interval': 300,       # frequent evals = more points on the curve
}

run_scaling(scaling_args)

In [ ]:
display(Image('plots/scaling_laws.png'))

## 7. Attention Head Visualization

Inspect what each attention head has learned by plotting the attention weight
heatmaps for every layer of the best model.

Bright cells mean the token in that row attended strongly to the token in that
column. A well-trained causal model should show structured patterns — early
layers often track local context, later layers can pick up longer-range
dependencies like piece-move relationships.

In [ ]:
BEST_VARIANT = 'rope'   # same checkpoint used in Section 4

# Run a short prompt through the model to populate the attention weight cache
model.eval()
prompt_moves = ['e4', 'e5', 'Nf3', 'Nc6', 'Bb5']   # Ruy Lopez opening
prompt_ids = tokenizer.encode(prompt_moves, add_special=False)
idx = torch.tensor([prompt_ids], device=device)

with torch.no_grad():
    model(idx)

# Save one PNG per layer to plots/attn/
os.makedirs('plots/attn', exist_ok=True)
plot_all_layers(model, tokenizer, idx, out_dir='plots/attn', title_prefix=BEST_VARIANT.upper())

# Display layer 0 inline
display(Image('plots/attn/attn_layer0.png'))
display(Image('plots/attn/attn_layer1.png'))
